# NeuraSight — Chest X-Ray Training (Kermany Dataset)

**Dataset:** Kermany et al. Chest X-ray (Normal / Bacterial Pneumonia / Viral Pneumonia)
**Platform:** AWS SageMaker (or Google Colab)
**Target Accuracy:** 93-96%

Key features:
- Single-source clean dataset (Guangzhou hospital)
- 3 classes split from filenames
- Proper 70/15/15 stratified split
- Class weighting + WeightedRandomSampler
- 30 epochs with early stopping
- Stacking ensemble included

In [ ]:
import os, zipfile, shutil, pathlib

# Kaggle setup
os.environ['KAGGLE_USERNAME'] = 'YOUR_USERNAME'
os.environ['KAGGLE_KEY'] = 'YOUR_KEY'
kaggle_dir = pathlib.Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
if not kaggle_json.exists():
    import json as _j
    with open(kaggle_json, 'w') as f:
        _j.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
    os.chmod(str(kaggle_json), 0o600)

!pip install kaggle timm seaborn scikit-learn -q

# Download dataset
LOCAL_DATA = '/tmp/chest_xray_kermany'
RAW_DIR = '/tmp/kermany_raw/chest_xray'
if not os.path.isdir(os.path.join(LOCAL_DATA, 'train')):
    print('Downloading Kermany Chest X-ray dataset...')
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /tmp/ --force
    print('Extracting...')
    with zipfile.ZipFile('/tmp/chest-xray-pneumonia.zip', 'r') as zf:
        zf.extractall('/tmp/kermany_raw')
    os.remove('/tmp/chest-xray-pneumonia.zip')
    print('\u2713 Extracted')
else:
    print('\u2713 Dataset already prepared')

In [ ]:
import random
from sklearn.model_selection import train_test_split
import numpy as np
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RAW_DIR = '/tmp/kermany_raw/chest_xray'

# Collect ALL images from all splits (train+val+test) into one pool
all_images = []  # (path, class_name)

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(RAW_DIR, split)
    if not os.path.isdir(split_dir):
        continue

    # NORMAL class
    normal_dir = os.path.join(split_dir, 'NORMAL')
    if os.path.isdir(normal_dir):
        for f in os.listdir(normal_dir):
            if f.lower().endswith(('.jpeg', '.jpg', '.png')):
                all_images.append((os.path.join(normal_dir, f), 'normal'))

    # PNEUMONIA class — split by filename
    pneumonia_dir = os.path.join(split_dir, 'PNEUMONIA')
    if os.path.isdir(pneumonia_dir):
        for f in os.listdir(pneumonia_dir):
            if f.lower().endswith(('.jpeg', '.jpg', '.png')):
                if 'bacteria' in f.lower():
                    all_images.append((os.path.join(pneumonia_dir, f), 'bacteria'))
                elif 'virus' in f.lower():
                    all_images.append((os.path.join(pneumonia_dir, f), 'virus'))

print(f'Total images collected: {len(all_images)}')
labels = [img[1] for img in all_images]
print(f'Class distribution: {Counter(labels)}')

# Stratified 70/15/15 split
CLASS_NAMES = ['bacteria', 'normal', 'virus']
paths = [img[0] for img in all_images]
labels_idx = [CLASS_NAMES.index(l) for l in labels]

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels_idx, test_size=0.3, random_state=SEED, stratify=labels_idx)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels)

print(f'Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}')

# Copy into organized folder structure
LOCAL_DATA = '/tmp/chest_xray_kermany'
ORGANIZED_DIR = LOCAL_DATA

for split_name, split_paths, split_labels in [
    ('train', train_paths, train_labels),
    ('val', val_paths, val_labels),
    ('test', test_paths, test_labels),
]:
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(ORGANIZED_DIR, split_name, cls), exist_ok=True)

    for path, label in zip(split_paths, split_labels):
        cls = CLASS_NAMES[label]
        dst = os.path.join(ORGANIZED_DIR, split_name, cls, os.path.basename(path))
        if not os.path.exists(dst):
            shutil.copy2(path, dst)

# Verify
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(ORGANIZED_DIR, split)
    classes = sorted(os.listdir(split_dir))
    counts = {c: len(os.listdir(os.path.join(split_dir, c))) for c in classes}
    print(f'{split}: {counts}')

In [ ]:
import os, json, time, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, accuracy_score
)
from sklearn.linear_model import LogisticRegression
import pickle

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

LOCAL_DATA = '/tmp/chest_xray_kermany'
TRAIN_DIR = os.path.join(LOCAL_DATA, 'train')
VAL_DIR = os.path.join(LOCAL_DATA, 'val')
TEST_DIR = os.path.join(LOCAL_DATA, 'test')
SAVE_DIR = '/tmp/outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_CONFIG = {
    "efficientnet": {"timm_name": "efficientnet_b0", "save_name": "CHEST_XRAY_EFFICIENTNET"},
    "resnet": {"timm_name": "resnet50", "save_name": "CHEST_XRAY_RESNET"},
    "densenet": {"timm_name": "densenet121", "save_name": "CHEST_XRAY_DENSENET"},
}
MODELS_TO_TRAIN = ["efficientnet", "resnet", "densenet"]
NUM_CLASSES = 3
DISPLAY_NAMES = ["Bacteria", "Normal", "Virus"]  # alphabetical from ImageFolder

EPOCHS = 30
PATIENCE = 7
LR = 3e-5
WEIGHT_DECAY = 1e-3
BATCH_SIZE = 32
LABEL_SMOOTHING = 0.1

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = ImageFolder(TEST_DIR, transform=val_transform)

print(f'Classes: {train_dataset.classes}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

# Class weights for loss
class_counts = np.bincount([label for _, label in train_dataset.samples])
total = sum(class_counts)
class_weights = torch.tensor(
    [total / (NUM_CLASSES * c) for c in class_counts], dtype=torch.float32
).to(device)
print(f'Class weights: {class_weights.cpu().numpy()}')

# WeightedRandomSampler for balanced batches
sample_weights = [1.0 / class_counts[label] for _, label in train_dataset.samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

In [ ]:
def create_model(timm_name):
    """Create a timm model with custom classifier head + dropout."""
    model = timm.create_model(timm_name, pretrained=True, num_classes=0)  # no head
    num_features = model.num_features
    classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(num_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, NUM_CLASSES),
    )
    model = nn.Sequential(model, classifier)
    return model.to(device)


def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    return (running_loss / total, 100.0 * correct / total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

In [ ]:
def train_full(model_key):
    """Train a single model end-to-end."""
    cfg = MODEL_CONFIG[model_key]
    print(f'\n{"="*60}')
    print(f'Training: {model_key.upper()} ({cfg["timm_name"]})')
    print(f'{"="*60}')

    model = create_model(cfg['timm_name'])
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_acc = 0.0
    patience_counter = 0
    best_path = os.path.join(SAVE_DIR, cfg['save_name'] + '.pth')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()
        elapsed = time.time() - t0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f'  Epoch {epoch+1:02d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | '
              f'{elapsed:.1f}s')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    # Load best and evaluate on test set
    model.load_state_dict(torch.load(best_path))
    test_loss, test_acc, preds, labels, probs = evaluate(model, test_loader, criterion)

    # Save test probabilities for ensemble
    probs_path = os.path.join(SAVE_DIR, cfg['save_name'] + '_test_probs.npy')
    np.save(probs_path, probs)

    # Save test labels (same for all models)
    labels_path = os.path.join(SAVE_DIR, 'test_labels.npy')
    if not os.path.exists(labels_path):
        np.save(labels_path, labels)

    # Metrics
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    print(f'\n  TEST Results for {model_key}:')
    print(f'    Accuracy:  {test_acc:.2f}%')
    print(f'    Precision: {precision:.4f}')
    print(f'    Recall:    {recall:.4f}')
    print(f'    F1-Score:  {f1:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(labels, preds, target_names=DISPLAY_NAMES))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES, ax=axes[0])
    axes[0].set_title(f'{model_key} — Confusion Matrix')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

    # Training curves
    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'], label='Val Acc')
    axes[1].set_title(f'{model_key} — Training Curves')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, cfg['save_name'] + '_plots.png'), dpi=100)
    plt.show()

    # Save locally (download later)
    print(f'  Model saved to: {best_path}')
    print(f'  Probs saved to: {probs_path}')

    metrics = {
        'model': model_key,
        'accuracy': test_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'best_val_acc': best_val_acc,
    }

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return metrics

In [ ]:
all_metrics = []
for model_key in MODELS_TO_TRAIN:
    metrics = train_full(model_key)
    all_metrics.append(metrics)

df = pd.DataFrame(all_metrics)
print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
print(df[['model', 'accuracy', 'precision', 'recall', 'f1']].to_string(index=False))

In [ ]:
# Load probabilities from all 3 models
probs = {}
for key in MODELS_TO_TRAIN:
    save_name = MODEL_CONFIG[key]['save_name']
    probs[key] = np.load(os.path.join(SAVE_DIR, save_name + '_test_probs.npy'))
    print(f'{key}: {probs[key].shape}')

y_true = np.load(os.path.join(SAVE_DIR, 'test_labels.npy'))
print(f'Labels: {y_true.shape}, distribution: {np.bincount(y_true.astype(int))}')

# Concatenate: (N, 9) feature matrix
X = np.hstack([probs[key] for key in MODELS_TO_TRAIN])
print(f'Feature matrix: {X.shape}')

# 50/50 leakage-safe split for meta-learner
from sklearn.model_selection import train_test_split as tts2
X_meta_train, X_meta_test, y_meta_train, y_meta_test = tts2(
    X, y_true, test_size=0.5, random_state=SEED, stratify=y_true)

# Train meta-learner
meta = LogisticRegression(max_iter=1000, random_state=SEED, multi_class='multinomial')
meta.fit(X_meta_train, y_meta_train)

# Evaluate ensemble
ensemble_pred = meta.predict(X_meta_test)
ens_acc = accuracy_score(y_meta_test, ensemble_pred) * 100
print(f'\nENSEMBLE Accuracy: {ens_acc:.2f}%')
print(classification_report(y_meta_test, ensemble_pred, target_names=DISPLAY_NAMES))

# Ensemble confusion matrix
cm_ens = confusion_matrix(y_meta_test, ensemble_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Greens',
            xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES)
plt.title('Stacking Ensemble — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'ENSEMBLE_confusion_matrix.png'), dpi=100)
plt.show()

# Compare with base models on same split
print('\nCOMPARISON (same meta-test split):')
for i, key in enumerate(MODELS_TO_TRAIN):
    base_pred = X_meta_test[:, i*NUM_CLASSES:(i+1)*NUM_CLASSES].argmax(1)
    base_acc = accuracy_score(y_meta_test, base_pred) * 100
    print(f'  {key:12s}: {base_acc:.2f}%')
print(f'  {"ENSEMBLE":12s}: {ens_acc:.2f}%')

# Save meta-learner
meta_path = os.path.join(SAVE_DIR, 'meta_model_Chest_Xray.pkl')
with open(meta_path, 'wb') as f:
    pickle.dump(meta, f)

# Save config
config = {
    "model_order": MODELS_TO_TRAIN,
    "save_names": {k: MODEL_CONFIG[k]['save_name'] for k in MODELS_TO_TRAIN},
    "class_names": DISPLAY_NAMES,
    "num_classes": NUM_CLASSES,
    "meta_learner": "LogisticRegression",
    "feature_dim": NUM_CLASSES * len(MODELS_TO_TRAIN),
    "ensemble_accuracy": round(ens_acc, 2)
}
config_path = os.path.join(SAVE_DIR, 'chest_xray_ensemble_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'\n\u2713 Ensemble saved: {meta_path}')
print(f'\u2713 Config saved: {config_path}')

In [ ]:
# List all outputs
print('Files to download:')
print('-' * 40)
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024*1024)
    print(f'  {f} ({size:.1f} MB)')

# Create download links (works in both SageMaker and Colab)
try:
    from google.colab import files as colab_files
    for f in os.listdir(SAVE_DIR):
        colab_files.download(os.path.join(SAVE_DIR, f))
    print('\n\u2713 Files downloading via Colab...')
except ImportError:
    # SageMaker — zip for easy download
    import zipfile as zf2
    zip_path = '/tmp/chest_xray_outputs.zip'
    with zf2.ZipFile(zip_path, 'w', zf2.ZIP_DEFLATED) as z:
        for f in os.listdir(SAVE_DIR):
            z.write(os.path.join(SAVE_DIR, f), f)
    zip_size = os.path.getsize(zip_path) / (1024*1024)
    print(f'\n\u2713 All files zipped: {zip_path} ({zip_size:.1f} MB)')
    print('Download from the file browser (left panel) \u2192 /tmp/chest_xray_outputs.zip')

    from IPython.display import FileLink, display
    display(FileLink(zip_path))

In [ ]:
print('='*60)
print('TRAINING COMPLETE')
print('='*60)
print(f'\nDataset: Kermany Chest X-ray (Normal / Bacteria / Virus)')
print(f'Total images: ~5,856 (70/15/15 split)')
print(f'Models: EfficientNet-B0, ResNet-50, DenseNet-121')
print(f'Ensemble: Logistic Regression Stacking')
print(f'\nResults:')
for m in all_metrics:
    print(f"  {m['model']:12s}: {m['accuracy']:.2f}%")
print(f"  {'ENSEMBLE':12s}: {ens_acc:.2f}%")
print(f'\nFiles saved in: {SAVE_DIR}')
print('\nNext steps:')
print('  1. Download chest_xray_outputs.zip')
print('  2. Extract .pth + .pkl + .json files')
print('  3. Place them in NeuraSight/backend/fastapi/app/modules/chest_xray/weights/')
print('  4. Update config.py with new model paths')

In [ ]:
# Optional: cleanup raw data to free disk space
# Uncomment to run:
# shutil.rmtree('/tmp/kermany_raw', ignore_errors=True)
# shutil.rmtree(LOCAL_DATA, ignore_errors=True)
# print('\u2713 Cleaned up temporary files')
print('Notebook execution complete. Download your outputs above.')